In [ ]:
import random
import re
import matplotlib.pyplot as plt
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Movie Class
class Movie:
    def __init__(self, movie_id, title,  seats):
        self.movie_id = movie_id
        self.title = title
        self.seats = seats   
        self.reviews = [] 
        self.earnings = 0



# Customer Class
class Customer:
    def __init__(self, username, password, email, phone):
        self.username = username
        self.password = password
        self.email = email
        self.phone = phone

# Main System Class
class TicketBookingSystem:
    def __init__(self):
        self.movies = {}   # movie_id -> Movie
        self.customers = {}  # username -> Customer object
        self.bookings = {}  # ticket_id -> (customer_name, movie_title)
        self.reviews = {} 
        self.load_data()

    # Save Movies
    def save_movies(self):
        with open("movies.txt", "w") as file:
            for movie in self.movies.values():
                seats_data = f"{movie.seats['Premium']}:{movie.seats['Standard']}"
                file.write(f"{movie.movie_id},{movie.title},{seats_data},{movie.earnings}\n")

    # Save Customers
    def save_customers(self):
        with open("customers.txt", "w") as file:
            for customer in self.customers.values():
                file.write(f"{customer.username},{customer.password},{customer.email},{customer.phone}\n")
    def save_reviews(self):
        with open("reviews.txt", "w") as file:
            for movie_id, review_list in self.reviews.items():
                for username, rating, comment in review_list:
                    file.write(f"{movie_id},{username},{rating},{comment}\n")
    def save_bookings(self):
        with open("bookings.txt", "w") as file:
            for ticket_id, (username, movie_title, num_tickets, seat_type) in self.bookings.items():
                file.write(f"{ticket_id},{username},{movie_title},{num_tickets},{seat_type}\n")

    # Load Data from Files
    def load_data(self):
    
        try:
            with open("movies.txt", "r") as file:
                for line in file:
                    parts = line.strip().split(",")
    
                    if len(parts) < 4:  # Ensure correct format
                        print(f"Skipping invalid movie entry: {line.strip()}")
                        continue  
    
                    movie_id = parts[0]
                    title = ",".join(parts[1:-2])  # Merge title if it has commas
                    seats_data = parts[-2]
                    earnings = parts[-1]
    
                    # Convert seat data properly
                    premium, standard = map(int, seats_data.split(":"))
                    self.movies[int(movie_id)] = Movie(int(movie_id), title, {
                        "Premium": [f"P{i+1} - available" for i in range(premium)],
                        "Standard": [f"S{i+1} - available" for i in range(standard)]
                    }, int(earnings))
    
        except FileNotFoundError:
            print("movies.txt not found! Starting with an empty movie list.")
        except Exception as e:
            print(f"Error loading data: {e}")
        try:
            with open("customers.txt", "r") as file:
                for line in file:
                    username, password, email, phone = line.strip().split(",")
                    self.customers[username] = Customer(username, password, email, phone)
        except FileNotFoundError:
            pass

        
        try:
            with open("reviews.txt", "r") as file:
                for line in file:
                    movie_id, username, rating, comment = line.strip().split(",", 3)
                    movie_id = int(movie_id)
                    rating = float(rating)
                    if movie_id not in self.reviews:
                        self.reviews[movie_id] = []
                    self.reviews[movie_id].append((username, rating, comment))
        except FileNotFoundError:
            pass
        try:
            with open("bookings.txt", "r") as file:
                for line in file:
                    parts = line.strip().split(",")
                    if len(parts) != 5:
                        print(f"Skipping invalid booking entry: {line.strip()}")
                        continue  # Skip improperly formatted lines
                    
                    ticket_id, username, movie_title, num_tickets, seats_data = parts
                    self.bookings[int(ticket_id)] = (username, movie_title, int(num_tickets), seats_data)
        except FileNotFoundError:
            pass
    # Admin Functions
    def set_movie(self, title,  premium_seats, standard_seats):
        movie_id = random.randint(1000, 9999)
        self.movies[movie_id] = Movie(movie_id, title, {'Premium': premium_seats, 'Standard': standard_seats})
        self.save_movies()
        self.initialize_seats(movie_id, premium_seats, standard_seats)
        print(f"Movie '{title}' added successfully with ID: {movie_id}")

    def delete_movie(self, movie_id):
        if movie_id in self.movies:
            del self.movies[movie_id]
            self.save_movies()
            print("Movie deleted successfully!")
        else:
            print("Invalid Movie ID!")

    # Customer Functions
    def register_customer(self, username, password, email, phone):
        if username in self.customers:
            print("Username already exists! Choose another.")
        else:
            self.customers[username] = Customer(username, password, email, phone)
            self.save_customers()
            print("Customer registered successfully!")

    def login_customer(self, username, password):
        return username in self.customers and self.customers[username].password == password

    def view_profile(self, username):
        if username in self.customers:
            customer = self.customers[username]
            print(f"\nCustomer Profile:\nUsername: {customer.username}\nEmail: {customer.email}\nPhone: {customer.phone}")
        else:
            print("Customer not found!")

    def update_profile(self, username):
        if username in self.customers:
            email = input("Enter new email: ")
            phone = input("Enter new phone number: ")
            if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", email):
                print("Invalid email format! Please enter a valid email address.")
            elif not re.match(r"^\d{10}$", phone):
                print("Invalid phone number! Please enter a 10-digit phone number.")
            else:
                self.customers[username].email = email
                self.customers[username].phone = phone
                self.save_customers()
                print("Profile updated successfully!")
        else:
            print("Customer not found!")

    def change_password(self, username):
        if username in self.customers:
            new_password = input("Enter new password: ")
            if not re.match(r"^(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$",new_password):
                print('Invalid password! Please use one special character and digit with minimum 8 length')
            else:
                self.customers[username].password = new_password
                self.save_customers()
                print("Password changed successfully!")
        else:
            print("Customer not found!")

    def view_movies(self):
        if not self.movies:
            print("No movies available.")
            return
        print("\nAvailable Movies:")
        for movie in self.movies.values():
            print(f"Movie_ID:{movie.movie_id} - Movie_Name:{movie.title} ")

    
    def send_email(self,receiver_email, subject, body):
        sender_email = "hetvi7669@gmail.com"  # Replace with your email
        sender_password = "ebqb sjtv fqvi iaaq"  # Replace with your app password
    
        # Create the email message
        msg = MIMEMultipart()
        msg["From"] = sender_email
        msg["To"] = receiver_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "html"))
    
        try:
            # Establish connection with the email server
            server = smtplib.SMTP("smtp.gmail.com", 587)
            server.starttls()
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, receiver_email, msg.as_string())
            server.quit()
            print("Email sent successfully!")
        except Exception as e:
            print(f"Error sending email: {e}")
    def booking_confirmation_email(self,ticket_id, movie_name, no_of_seats, seat_type, receiver_email,total_price):
        
        subject = "Movie Ticket Booking Confirmation"
        body = f"""
        <html>
        <body>
            <h2>Booking Confirmation</h2>
            <p>Your ticket has been successfully booked!</p>
            <p><strong>Movie:</strong> {movie_name}</p>
            <p><strong>Ticket ID:</strong> {ticket_id}</p>
            <p><strong>Number of Seats:</strong> {no_of_seats}</p>
            <p><strong>Seat Type:</strong> {seat_type}</p>
            <p><strong>Total Amount:</strong> {total_price}</p>
            <p>Enjoy your movie!</p>
        </body>
        </html>
        """
        self.send_email(receiver_email, subject, body)

    # Book Ticket Function
    def initialize_seats(self, movie_id, premium_count, standard_count):
        if movie_id not in self.movies:
            print("Invalid Movie ID!")
            return
    
        self.movies[movie_id].seats = {
            "Premium": [f"P{i+1} - available" for i in range(premium_count)],
            "Standard": [f"S{i+1} - available" for i in range(standard_count)]
        }

    def book_ticket(self, username, movie_id, seat_type, num_tickets):
        if movie_id not in self.movies:
            print("Invalid Movie ID!")
            return
    
        # Convert seat type input
        seat_type = seat_type.lower()
        if seat_type == 'p':
            seat_type = 'Premium'
        elif seat_type == 's':
            seat_type = 'Standard'
        else:
            print("Invalid seat type! Please enter 'p' for Premium or 's' for Standard.")
            return
    
        movie = self.movies[movie_id]
        price_chart = {'Premium': 300, 'Standard': 150}
    
        # Ensure seats are stored as a list
        if not isinstance(movie.seats[seat_type], list):
            print("Seat data is not initialized correctly!")
            return
    
        available_seats = [seat for seat in movie.seats[seat_type] if "available" in seat]
    
        if len(available_seats) < num_tickets:
            print(f"Not enough {seat_type} seats available!")
            return
    
        print(f"Available {seat_type} seats: {', '.join(available_seats)}")
    
        chosen_seats = []
        while len(chosen_seats) < num_tickets:
            seat_choice = input(f"Choose your seat ({len(chosen_seats) + 1}/{num_tickets}): ").strip()

            matching_seat = next((seat for seat in available_seats if seat.startswith(seat_choice + " - available")), None)
            
            if matching_seat:
                chosen_seats.append(matching_seat.split(" - ")[0])  
                available_seats.remove(matching_seat)  # Mark seat as booked
                movie.seats[seat_type][movie.seats[seat_type].index(matching_seat)] = matching_seat.replace("available", "booked")
            else:
                print("Invalid or already booked seat. Choose again.")    
        total_price = num_tickets * price_chart[seat_type]
        movie.earnings += total_price
        ticket_id = random.randint(10000, 99999)
        while ticket_id in self.bookings:
            ticket_id = random.randint(10000, 99999)
    
        self.bookings[ticket_id] = (username, movie.title, chosen_seats, seat_type)
    
        receiver_email = input("Enter Your Email For Confirmation Of Booking: ")
        if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", receiver_email):
            print("Invalid email format! Please enter a valid email address.")
        else:
            self.booking_confirmation_email(ticket_id, movie.title, num_tickets, seat_type, receiver_email,total_price)
            print(f"Ticket booked successfully! Ticket ID: {ticket_id}")
            print(f"Booked seats: {', '.join(chosen_seats)} for {movie.title}. Total cost: Rs. {total_price}")
            self.save_movies()
            self.save_bookings()
    def cancel_booking(self, ticket_id):
        if ticket_id not in self.bookings:
            print("Invalid Ticket ID!")
            return    
    
        username, movie_title, booked_seats, seat_type = self.bookings[ticket_id]
        
        # Find the corresponding movie
        movie = next((m for m in self.movies.values() if m.title == movie_title), None)        
        if not movie:
            print("Movie not found!")  
            return    
    
        price_chart = {'Premium': 300, 'Standard': 150}
        
        # Ensure seat type is valid
        if seat_type not in movie.seats:
            print(f"Invalid seat type: {seat_type}")
            return
    
        # Convert booked seat names to "P1 - booked" or "S1 - booked"
        booked_seat_names = [f"{seat} - booked" for seat in booked_seats]
    
        for seat in booked_seat_names:
            if seat in movie.seats[seat_type]:
                seat_index = movie.seats[seat_type].index(seat)
                movie.seats[seat_type][seat_index] = seat.replace("booked", "available")  # Make seat available again
            else:
                print(f"Warning: {seat} not found in the system!")
    
        movie.earnings -= len(booked_seats) * price_chart[seat_type]
        total_price = len(booked_seats) * price_chart[seat_type]
        # Remove booking record
        del self.bookings[ticket_id]
    
        receiver_email = input("Enter Your Email For Cancellation Of Booking: ")
        if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", receiver_email):
            print("Invalid email format! Please enter a valid email address.")
        else:
            self.booking_cancellation_email(ticket_id, movie.title, len(booked_seats), seat_type, receiver_email,total_price)
            self.save_movies()
            self.remove_booking_from_file(ticket_id)
            
            print(f"Booking canceled successfully! Ticket ID: {ticket_id}")
            print(f"Movie: {movie.title}, Seats Returned: {', '.join(booked_seats)}")

    def check_availability(self, movie_id, seat_type):
        if movie_id not in self.movies:
            print("Invalid Movie ID!")
            return
        
        movie = self.movies[movie_id]
        if seat_type not in movie.seats:
            print(f"Invalid seat type: {seat_type}")
            return
        
        seat_status = ', '.join(movie.seats[seat_type])
        print(f"Seat availability for {movie.title} ({seat_type} seats):")
        print(seat_status)

    def booking_cancellation_email(self,ticket_id, movie_name, no_of_seats, seat_type,receiver_email,total_price):
        subject = "Movie Ticket Cancellation"
        body = f"""
        <html>
        <body>
            <h2>Booking Cancellation</h2>
            <p>Your ticket has been cancelled.</p>
            <p><strong>Movie:</strong> {movie_name}</p>
            <p><strong>Ticket ID:</strong> {ticket_id}</p>
            <p><strong>Number of Seats:</strong> {no_of_seats}</p>
            <p><strong>Seat Type:</strong> {seat_type}</p>
            <p><strong>Amouut Returned:</strong> {total_price}</p>
            <p>We hope to see you again!</p>
        </body>
        </html>
        """
        self.send_email(receiver_email, subject, body)

    def remove_booking_from_file(self, ticket_id):
        """Removes a booking from bookings.txt by rewriting the file without the canceled entry."""
        try:
            with open("bookings.txt", "r") as file:
                lines = file.readlines()

            with open("bookings.txt", "w") as file:
                for line in lines:
                    if not line.startswith(str(ticket_id) + ","):
                        file.write(line)
        except FileNotFoundError:
            print("Bookings file not found.")

    def add_review(self, username, movie_id):
        if movie_id not in self.movies:
            print("Invalid Movie ID!")
            return

        rating = input("Enter your rating (1-5): ")
        if not rating.isdigit() or not (1 <= int(rating) <= 5):
            print("Invalid rating! Please enter a number between 1 and 5.")
            return
        rating = int(rating)

        comment = input("Enter your review: ")

        if movie_id not in self.reviews:
            self.reviews[movie_id] = []
        self.reviews[movie_id].append((username, rating, comment))

        self.save_reviews()
        print("Review added successfully!")

    # View Movie Reviews
    def view_reviews(self, movie_id):
        if movie_id not in self.movies:
            print("Invalid Movie ID!")
            return

        if movie_id not in self.reviews or not self.reviews[movie_id]:
            print("No reviews available for this movie.")
            return

        print(f"\nReviews for {self.movies[movie_id].title}:")
        total_rating = 0
        for username, rating, comment in self.reviews[movie_id]:
            total_rating += rating
            print(f"\nUser: {username}\nRating: {rating}/5\nReview: {comment}")

        avg_rating = total_rating / len(self.reviews[movie_id])
        print(f"\nAverage Rating: {avg_rating:.1f}/5")
    def display_movie_collections(self):
        movies = list(self.movies.values())
        titles = [movie.title for movie in movies]
        earnings = [movie.earnings for movie in movies]
        
        plt.figure(figsize=(5, 5))
        plt.bar(titles, earnings, color=['blue', 'red', 'green'])
        plt.xlabel('Movies')
        plt.ylabel('Total Collection (Rs.)')
        plt.title('Movie Earnings Chart')
        plt.xticks(rotation=45)
        plt.show()


# Admin Menu
def admin_menu(system):
    while True:
        print("\n--- Admin Menu ---")
        print("1. Set Movie")
        
        print("2. Delete Movie")
        print("3. View Movie")
        print("4. Logout")
        choice = input("Enter your choice: ")
        if choice == "1":
            title = input("Enter movie title: ")
            p_seats=int(input("Enter numberof premium seats: "))
            s_seats = int(input("Enter number of standard seats: "))
            system.set_movie(title, p_seats,s_seats)
        elif choice=="2":
            system.view_movies()
            
            movie_id = int(input("Enter Movie ID to delete: "))
            system.delete_movie(movie_id)
        elif choice == "3":
            system.view_movies()
        elif choice == "4":
            print("Logging out...")
            break
        else:
            print("Invalid choice!")

# Customer Menu
def customer_menu(system, username):
    while True:
        print("\n--- Customer Menu ---")
        print("1. View Profile")
        print("2. Update Profile")
        print("3. Change Password")
        print("4. View Movies")
        print("5. Book Ticket")
        print('6. Cancel Booking')
        print("7. Add Movie Review & Rating")
        print("8. View Movie Reviews")
        print("9. Movie collection Ration")
        print("10. Logout")
        choice = input("Enter your choice: ")
        if choice == "1":
            system.view_profile(username)
        elif choice == "2":
            system.update_profile(username)
        elif choice == "3":
            system.change_password(username)
        elif choice == "4":
            system.view_movies()
        elif choice == "5":
            system.view_movies()
            with open("movies.txt",'r') as file :
                movie=file.readlines()
                length=len(movie)
            if length==0:
                return customer_menu(system,username)
            
            movie_id = int(input("Enter Movie ID to book: "))
            num_tickets = int(input("Enter number of tickets: "))
            seat_type=input("Enter seat type p/s")
            system.book_ticket(username, movie_id,seat_type ,num_tickets)
        elif choice=='6':
            with open("bookings.txt", "r") as file:
                bookt=file.readlines()
                length=len(bookt)
                if length == 0:
                    print("No bookings available")
                    return customer_menu(system,username)
                for i in bookt:
                    a=i.split(',')
                    if username == a[1]:
                        print('Ticket Id:',a[0])
                        print('Movie You Booked :',a[2])
                        
            ticket_id=int(input("Enter Ticket Id for cancel booking: "))
            system.cancel_booking(ticket_id)
        elif choice == "7":
            system.view_movies()
            movie_id = int(input("Enter Movie ID to review: "))
            system.add_review(username, movie_id)
        elif choice == "8":
            system.view_movies()
            movie_id = int(input("Enter Movie ID to view reviews: "))
            system.view_reviews(movie_id)
        elif choice=='9':
            system.display_movie_collections()
        elif choice == "10":
            print("Logging out...")
            break
        else:
            print("Invalid choice!")
# Main Function
def main():
    system = TicketBookingSystem()
    while True:
        print("\nWelcome to Movie Ticket System")
        print("1. Admin Login")
        print("2. Customer Login")
        print("3. Register New Customer")
        print("4. Exit")
        user_choice = input("Enter your choice: ")

        if user_choice == "1":
            password = input("Enter admin password: ")
            if password == "admin123":
                admin_menu(system)
            else:
                print("Incorrect password!")
        elif user_choice == "2":
            username = input("Enter username: ")
            password = input("Enter password: ")
            if system.login_customer(username, password):
                customer_menu(system, username)
                
            else:
                print("Invalid username or password!")
        elif user_choice == "3":
            username = input("Enter new username: ")
            password = input("Enter new password: ")
            email = input("Enter email: ")
            phone = input("Enter phone number: ")
            if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", email):
                print("Invalid email format! Please enter a valid email address.")
            elif not re.match(r"^\d{10}$", phone):
                print("Invalid phone number! Please enter a 10-digit phone number.")
            elif not re.match(r"^(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$",password):
                print('Invalid password! Please use one special character and digit with minimum 8 length')
            else:
                system.register_customer(username, password, email, phone)
        elif user_choice == "4":
            print("Thank you for using the system!")
            break
        else:
            print("Invalid choice!")

    
if __name__ == "__main__":
    main()


Welcome to Movie Ticket System
1. Admin Login
2. Customer Login
3. Register New Customer
4. Exit


Enter your choice:  1
Enter admin password:  admin123



--- Admin Menu ---
1. Set Movie
2. Delete Movie
3. View Movie
4. Logout


Enter your choice:  1
Enter movie title:  rrr
Enter numberof premium seats:  5
Enter number of standard seats:  5


Movie 'rrr' added successfully with ID: 6013

--- Admin Menu ---
1. Set Movie
2. Delete Movie
3. View Movie
4. Logout


...

Enter your choice:  5



Available Movies:
Movie_ID:5104 - Movie_Name:bahubali 


Enter Movie ID to book:  5104
Enter number of tickets:  2
Enter seat type p/s p


Available Premium seats: P1 - available, P2 - available, P3 - available, P4 - available, P5 - available


Choose your seat (1/2):  P1
Choose your seat (2/2):  P2
